# マルチモーダルプロンプティングのための Gemini

## 学習目標


1. Gemini を使用してテキストプロンプトからテキストを生成する方法を学ぶ
2. さまざまな機能と設定オプションについて学ぶ
3. Gemini を使用して画像とテキストプロンプトからテキストを生成する方法を学ぶ
4. Gemini を使用して動画とテキストプロンプトからテキストを生成する方法を学ぶ
5. Gemini を使用してすべてのモダリティを同時に活用する方法を学ぶ



## 概要

### Gemini
このノートブックでは、Python 用 Google Gen AI SDK を使用して Gemini を操作する方法を学びます。
Gemini は、Google DeepMind によって開発された、マルチモーダルのユースケース向けに設計された生成 AI モデルファミリー（Gemini Pro、Gemini Flash、Gemini Flash-Lite など）です。

Gemini には、次のようなさまざまな方法でアクセスできます。
- 迅速なテストとコマンド生成を行うための [Agent Platform Studio](https://cloud.google.com/products/gemini-enterprise-agent-platform)
- cURL コマンド
- Google Gen AI SDK

### Google Gen AI SDK
[Google Gen AI Python SDK](https://github.com/googleapis/python-genai) は、デベロッパーが Gemini を含む Google の生成モデルを Python アプリケーションに統合するためのインターフェースを提供します。[Gemini Developer API](https://ai.google.dev/gemini-api/docs) と [Agent Platform API](https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/beginners-guide) の両方をサポートしています。


このノートブックでは、**Gen AI SDK** を使用して Agent Platform Gemini API を呼び出すことに焦点を当てています。

詳細については、[Agent Platform 上の生成 AI](https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/beginners-guide) のドキュメントをご覧ください。

## ライブラリのインポート

In [ ]:
import http.client
import io
import typing
import urllib.request

import IPython.display
from google import genai
from google.genai.types import GenerateContentConfig, Part, ThinkingConfig
from PIL import Image as PIL_Image
from PIL import ImageOps as PIL_ImageOps
from pydantic import BaseModel

## セットアップ
### Gemini のバージョンを指定

Gemini モデルは、自然言語タスク、マルチターンのテキストおよびコードチャット、コード生成を処理するように設計されています。ここでは `gemini-3.5-flash` バージョンを使用します。

In [ ]:
MODEL = "gemini-3.5-flash"

### Gen AI SDK クライアントのセットアップ
Gen AI SDK クライアントをインスタンス化しましょう。Gemini は Agent Platform または Gemini Developer API のいずれかを経由して呼び出すことができます。

ここでは `vertexai=True` を指定して Agent Platform を使用します。

In [ ]:
client = genai.Client(vertexai=True, location="global")

## テキストプロンプトからのテキスト生成

モデルにテキストプロンプトを送信します。Gemini モデルはストリーミングレスポンスメカニズムを提供しています。このアプローチでは、完全なレスポンスを待つ必要はなく、データが利用可能になり次第すぐにフラグメント（断片）の処理を開始できます。

In [ ]:
responses = client.models.generate_content_stream(
    model=MODEL, contents="Why is the sky blue?"
)
for response in responses:
    print(response.text, end="")

### 独自のプロンプトを試す

- What are the biggest challenges facing the healthcare industry?（ヘルスケア業界が直面している最大の課題は何ですか？）
- What are the latest developments in the automotive industry?（自動車業界における最新の動向は何ですか？）
- What are the biggest opportunities in retail industry?（小売業界における最大のビジネスチャンスは何ですか？）
- （独自のプロンプトを試してみてください！）

In [ ]:
prompt = """Create a numbered list of 10 items. Each item in the list should be a trend in the tech industry.

Each trend should be less than 5 words."""  # 独自のプロンプトを試してみてください

responses = client.models.generate_content_stream(model=MODEL, contents=prompt)

for response in responses:
    print(response.text, end="")

### Thinking level（思考レベル）

Thinking level（思考レベル）は、モデルが回答を生成する前に行う内部推論（思考）の深さを制御します。`ThinkingConfig` 経由で `thinking_level` を設定することで、推論のレベルを調整できます。オプションには `"MINIMAL"`、`"LOW"`、`"MEDIUM"`、`"HIGH"` があります。また、返される合計トークン数を制限するために `max_output_tokens` を指定することもできます。

`include_thoughts=True` を設定すると、モデルの内部推論プロセスを検査できます（`part.thought` が `True` の部分でアクセス可能）。

以下では、複数の制約を持つ複雑なタスク（頭文字が `S-T-A-R` の折句、各行正確に 8 音節、`AABB` の脚韻構成）を Gemini がどのように処理するかをテストします。
- **`MINIMAL`**: 最小限またはゼロの思考トークンで即座に生成します。高速で低レイテンシですが、厳格な制約（正確な音節数など）を見落とす可能性があります。
- **`HIGH`**: 内部推論トークンを割り当てて、最終的な出力を生成する前に計画、音節数の検証、自己修正を行います。

In [ ]:
PROMPT = """
Write a 4-line poem about space. 
Line 1 must start with 'S', line 2 with 'T', line 3 with 'A', line 4 with 'R'. 
Each line must have exactly 8 syllables and rhyme AABB.
"""

generation_config = GenerateContentConfig(
    thinking_config=ThinkingConfig(
        thinking_level="MINIMAL",
        include_thoughts=True
    ),
)

responses = client.models.generate_content_stream(
    model=MODEL, config=generation_config, contents=PROMPT
)

for chunk in responses:
    for candidate in chunk.candidates:
        if candidate.content and candidate.content.parts:
            for part in candidate.content.parts:
                if part.thought and part.text:
                    print(f"🤔 Thinking...: {part.text}", end="", flush=True)
                elif part.text:
                    print(f"{part.text}", end="", flush=True)

In [ ]:
generation_config = GenerateContentConfig(
    thinking_config=ThinkingConfig(
        thinking_level="HIGH",
        include_thoughts=True
    ),
)

responses = client.models.generate_content_stream(
    model=MODEL, config=generation_config, contents=PROMPT
)

for chunk in responses:
    for candidate in chunk.candidates:
        if candidate.content and candidate.content.parts:
            for part in candidate.content.parts:
                if part.thought and part.text:
                    print(f"🤔 Thinking...: {part.text}", end="", flush=True)
                elif part.text:
                    print(f"{part.text}", end="", flush=True)


### チャットプロンプトのテスト

Gemini モデルは自然なマルチターン会話をサポートしており、双方向の対話を必要とするテキストタスクに最適です。次の例は、マルチターン会話中にモデルがどのように応答するかを示しています。

In [ ]:
chat = client.chats.create(model=MODEL)
prompt = """My name is Ned. You are my personal assistant. My favorite movies are Lord of the Rings and Hobbit.

Suggest another movie I might like.
"""

responses = chat.send_message_stream(prompt)

for response in responses:
    print(response.text, end="")

このフォローアッププロンプトは、モデルが前のプロンプトに基づいてどのように応答するかを示しています。

In [ ]:
prompt = "Are my favorite movies based on a book series?"

responses = chat.send_message_stream(prompt)

for response in responses:
    print(response.text, end="")

チャット履歴を確認することもできます。

In [ ]:
print(chat.get_history())

## マルチモーダルデータに対する Gemini の活用

Gemini は、マルチモーダルプロンプトをサポートするマルチモーダルモデルです。プロンプトリクエストにテキスト、画像、動画を含めて、テキストやコードのレスポンスを受け取ることができます。

### ヘルパー関数の定義

画像を読み込んで表示するためのヘルパー関数を定義します。

In [ ]:
def display_byte_image(image_bytes: bytes) -> None:
    image = PIL_Image.open(io.BytesIO(image_bytes))
    IPython.display.display(image)


def get_image_bytes_from_url(image_url: str) -> bytes:
    if image_url.startswith("gs://"):
        image_url = get_url_from_gcs(image_url)
    with urllib.request.urlopen(image_url) as response:
        response = typing.cast(http.client.HTTPResponse, response)
        image_bytes = response.read()
    return image_bytes


def get_url_from_gcs(gcs_uri: str) -> str:
    # 画像表示用に GCS URI を URL に変換
    url = "https://storage.googleapis.com/" + gcs_uri.replace(
        "gs://", ""
    ).replace(" ", "%20")
    return url


def print_multimodal_prompt(contents: list):
    """Gemini に送信される contents を受け取り、

    可読性を高めるために完全なマルチモーダルプロンプトを出力します。
    """
    for content in contents:
        if isinstance(content, Part):
            if content.inline_data:
                mime_type = content.inline_data.mime_type
                if mime_type.startswith("image"):
                    display_byte_image(content.inline_data.data)
            elif content.file_data:
                mime_type = content.file_data.mime_type
                if mime_type.startswith("image"):
                    image_bytes = get_image_bytes_from_url(
                        content.file_data.file_uri
                    )
                    display_byte_image(image_bytes)
            elif content.text:
                print(content.text)
        if isinstance(content, str):
            print(content)

### テキストと画像からのテキスト生成
#### ローカル画像とテキストからのテキスト生成

`Part.from_bytes` メソッドを使用して、ローカルファイルをテキスト生成対象の画像として読み込みます。

In [ ]:
# Google Cloud Storage から画像をダウンロード
local_path = "image.jpg"
! gcloud storage cp "gs://cloud-samples-data/generative-ai/image/320px-Felis_catus-cat_on_snow.jpg" {local_path}

with open(local_path, "rb") as f:
    image_bytes = f.read()

# ローカルファイルから読み込み
image = Part.from_bytes(data=image_bytes, mime_type="image/jpeg")

# コンテンツを準備
prompt = (
    "Describe this image?"  # または Part.from_text(text="Describe this image?")
)

contents = [image, prompt]

responses = client.models.generate_content_stream(
    model=MODEL, contents=contents
)

print("-------Prompt--------")
print_multimodal_prompt(contents)

print("\n-------Response--------")
for response in responses:
    print(response.text, end="")

#### URL を持つ画像

画像が [Cloud Storage](https://docs.cloud.google.com/storage/docs) に保存されている場合や直接リンクがある場合は、プロンプトに含める画像の URI を指定できます。また、`mime_type` フィールドも指定する必要があります。画像でサポートされている MIME タイプには `image/png` や `image/jpeg` が含まれます。

Cloud Storage オブジェクトの URI（URL と混同しないでください）は、常に `gs://` で始まる必要があることに注意してください。

In [ ]:
# Cloud Storage URI から画像を読み込み
gcs_uri = "gs://cloud-samples-data/generative-ai/image/boats.jpeg"

# コンテンツを準備
image = Part.from_uri(file_uri=gcs_uri, mime_type="image/jpeg")
prompt = "Describe the scene?"
contents = [image, prompt]

responses = client.models.generate_content_stream(
    model=MODEL, contents=contents
)
print("-------Prompt--------")
print_multimodal_prompt(contents)

print("\n-------Response--------")
for response in responses:
    print(response.text, end="")

#### Few-shot プロンプティングのための複数の画像と JSON 形式テキストプロンプトの組み合わせ

一度に複数の画像を送信したり、テキストプロンプトと並べて任意の場所に画像を配置したりすることができます。

下の例では、Few-shot プロンプティングを実行して、特定の JSON 形式で都市とランドマークを Gemini に返させます。<br>
Pydantic モデルを提供することで、出力の JSON スキーマを強制できます。

In [ ]:
# Pydantic を使用して戻り値の JSON スキーマを定義
class LandmarkInfo(BaseModel):
    city: str
    landmark: str

In [ ]:
# Cloud Storage URI から画像を読み込み
image1_url = (
    "https://storage.googleapis.com/github-repo/img/gemini/intro/landmark1.jpg"
)
image2_url = (
    "https://storage.googleapis.com/github-repo/img/gemini/intro/landmark2.jpg"
)
image3_url = (
    "https://storage.googleapis.com/github-repo/img/gemini/intro/landmark3.jpg"
)
image1 = Part.from_uri(file_uri=image1_url, mime_type="image/jpeg")
image2 = Part.from_uri(file_uri=image2_url, mime_type="image/jpeg")
image3 = Part.from_uri(file_uri=image3_url, mime_type="image/jpeg")

# 出力例を準備
prompt1 = Part.model_dump_json(LandmarkInfo(city="London", landmark="Big Ben"))
prompt2 = Part.model_dump_json(LandmarkInfo(city="Paris", landmark="Eiffel Tower"))

# コンテンツを準備
contents = [image1, prompt1, image2, prompt2, image3]

# 出力スキーマを指定
generate_config = GenerateContentConfig(
    response_mime_type="application/json",
    response_schema=LandmarkInfo,
)

responses = client.models.generate_content_stream(
    model=MODEL, contents=contents, config=generate_config
)

print("-------Prompt--------")
print_multimodal_prompt(contents)

print("\n-------Response--------")
for response in responses:
    print(response.text, end="")

### 動画ファイルからのテキスト生成

プロンプトに含める動画の Cloud Storage URI を指定します。ファイルを保存するバケットは、リクエストを送信する Google Cloud プロジェクトと同じプロジェクト内にある必要があります。また、`mime_type` フィールドも指定する必要があります。動画でサポートされている MIME タイプには `video/mp4` が含まれます。

In [ ]:
file_path = "github-repo/img/gemini/multimodality_usecases_overview/pixel8.mp4"
video_uri = f"gs://{file_path}"
video_url = f"https://storage.googleapis.com/{file_path}"

IPython.display.Video(video_url, width=450)

In [ ]:
prompt = """
Answer the following questions using the video only:
What is the profession of the main person?
What are the main features of the phone highlighted?
Which city was this recorded in?
Provide the answer JSON.
"""

video = Part.from_uri(file_uri=video_uri, mime_type="video/mp4")
contents = [prompt, video]

generate_config = GenerateContentConfig(
    response_mime_type="application/json",
)

responses = client.models.generate_content_stream(
    model=MODEL, contents=contents, config=generate_config
)

for response in responses:
    print(response.text, end="")

### すべてのモダリティ（画像、動画、音声、テキスト）を同時に活用

Gemini はネイティブにマルチモーダルであり、異なるモダリティのデータのインターリーブ（交互配置）をサポートしています。同じ入力シーケンス内で、音声、視覚、テキスト、コードの入力を組み合わせてサポートできます。

In [ ]:
video_file_path = (
    "cloud-samples-data/generative-ai/video/behind_the_scenes_pixel.mp4"
)
video_file_uri = f"gs://{video_file_path}"
video_file_url = f"https://storage.googleapis.com/{video_file_path}"

IPython.display.Video(video_file_url, width=450)

In [ ]:
image_file_path = "cloud-samples-data/generative-ai/image/a-man-and-a-dog.png"
image_file_uri = f"gs://{image_file_path}"
image_file_url = f"https://storage.googleapis.com/{image_file_path}"

IPython.display.Image(image_file_url, width=450)

In [ ]:
video_file = Part.from_uri(file_uri=video_file_uri, mime_type="video/mp4")
image_file = Part.from_uri(file_uri=image_file_uri, mime_type="image/png")

prompt = """
  Look through each frame in the video carefully and answer the questions.
  Only base your answers strictly on what information is available in the video attached.
  Do not make up any information that is not part of the video and do not be too
  verbose, be to the point.

  Questions:
  - When is the moment in the image happening in the video? Provide a timestamp.
  - What is the context of the moment and what does the narrator say about it?
"""

contents = [video_file, image_file, prompt]

response = client.models.generate_content(model=MODEL, contents=contents)
print(response.text)

## 謝辞

このノートブックは、Polong Lin 氏が作成した [チュートリアル](https://github.com/GoogleCloudPlatform/generative-ai/commits/main/gemini/getting-started/intro_gemini_python.ipynb) を基に作成されています。

Copyright 2024 Google LLC

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

     https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.